<a href="https://colab.research.google.com/github/lloydakresi/ml_journey/blob/main/Sequential_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch

In [2]:
torch.manual_seed(42)

In [3]:
class Linear():
  def __init__(self, input_dim, output_dim, set_bias=False):
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.weights = torch.randn((input_dim, output_dim))
    self.bias = torch.ones((output_dim)) if set_bias else None

  def __call__(self, x):
    self.out = x @ self.weights + self.bias if self.bias is not None else x @ self.weights
    return self.out

  def __repr__(self):
    return f"Linear(input_dim={self.input_dim}, output_dim={self.output_dim})"

  def parameters(self):
    return [self.weights] + ([] if self.bias is None else [self.bias])


class Tanh():
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out

  def __repr__(self):
    return f"Tanh()"

  def parameters(self):
    return []

class Sequential():
  def __init__(self, layers):
    self.layers = layers
    self.params = []

  def __repr__(self):
    s = []
    for layer in self.layers:
      s.append(layer)
    return f"Sequential(\n{s}\n)"

  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    self.out = x
    return self.out

  def parameters(self):
    self.params = [p for layer in self.layers for p in layer.parameters()]
    return self.params

In [ ]:
class BasicRNN():
  def __init__(self, n_features, n_neurons):
    self.hidden = n_neurons
    self.input = n_features
    self.W_x = torch.randn((n_features, n_neurons))
    self.W_h = torch.randn((n_neurons, n_neurons))
    self.b_h = torch.zeros((n_neurons))

  def __call__(self, x, h_x=None):
    self.state = h_x or torch.zeros((x.shape[-2], self.hidden))

    output = []
    #would have to check this out
    for input in x:
      self.state = torch.tanh(
          input @ self.W_x +
          self.state @ self.W_h +
          self.b_h
      )
      output.append(self.state)

    return output, self.state


  def __repr__(self):
    return f"BasicRNN(n_inputs={self.input}, n_output={self.hidden})"


  def parameters(self):
    return [self.W_x, self.W_h, self.b_h]








In [ ]:
class BiDirectionalRNN():
  def __init__(self, n_hidden, n_in):
    self.f_rnn = BasicRNN(n_in, n_hidden)
    self.b_rnn = BasicRNN(n_in, n_hidden)
    self.n_hidden = n_hidden * 2

  def __call__(self, input, state=None):

    f_h, b_h = state if state else (None, None)
    input_reverse = torch.flip(input, dims=[0])
    b_h = torch.flip(b_h, [0]) if b_h else None

    output_f, state_f = self.f_rnn(input, f_h)
    output_b, state_b = self.b_rnn(input_reverse, b_h)

    output_f = torch.stack(output_f)
    output_b = torch.flip(torch.stack(output_b), [0])

    outputs = torch.cat((output_f, output_b), dim=-1)
    return outputs, (state_f, state_b)


  def parameters(self):
    return self.f_rnn.parameters() + self.b_rnn.parameters()

  def __repr__(self):
    return f"BiDirectionalRNN(n_hidden={self.n_hidden})"


In [4]:
from google.colab import drive
drive.mount("/content/drive")
dataset_path = "/content/drive/MyDrive/Electric_Production.csv"

Mounted at /content/drive


In [5]:
import pandas as pd
dataset = pd.read_csv(dataset_path)
prod_data = dataset["IPG2211A2N"]
prod_data = prod_data.values
prod_data.shape

(397,)

In [6]:
seq_len = 50
seqs = []
targets = []
idx = prod_data.shape[0] - seq_len

for i in range(idx):
  seqs.append(prod_data[i: i+seq_len])
  targets.append([prod_data[i+seq_len]])

seqs = torch.tensor(np.array(seqs), dtype=torch.float32)
targets = torch.tensor(np.array(targets), dtype=torch.float32)

In [7]:
seqs.shape
targets.shape


torch.Size([347, 1])

In [8]:
seqs = seqs.unsqueeze(-1)
seqs = seqs.permute(1, 0, 2)
seqs.shape

torch.Size([50, 347, 1])

In [ ]:
print(seqs)

print(targets)


In [ ]:
brnn = BiDirectionalRNN(10, 1)
blinear = Linear(10*2, 1)

In [32]:
a = (None, None)
bool(all(a))

False

In [10]:
import torch.nn.functional as F

In [44]:
class LSTMCell():
  def __init__(self, batch_size, n_features, n_hidden):
    self.n_hidden = n_hidden
    self.batch_size = batch_size
    self.n_features = n_features

    #forget gate params
    self.W_f = torch.randn((self.n_features + self.n_hidden, self.n_hidden))
    self.b_f = torch.zeros(self.n_hidden)

    #input gate params
    self.W_i = torch.randn((self.n_features + self.n_hidden, self.n_hidden))
    self.b_i = torch.zeros(self.n_hidden)

    #candidate hidden state params
    self.W_c = torch.randn((self.n_features + self.n_hidden, self.n_hidden))
    self.b_c = torch.zeros(self.n_hidden)

    #output gate params
    self.W_o = torch.randn((self.n_features + self.n_hidden, self.n_hidden))
    self.b_o = torch.zeros(self.n_hidden)


  def __call__(self, input, state=(None, None)):
    self.c, self.h_x = state if all(state) else (torch.ones((self.batch_size, self.n_hidden)),
                                 torch.zeros((self.batch_size, self.n_hidden)))
    long_term_outputs = []
    short_term_outputs = []

    for x in input:
      self.in_x = torch.concat((self.h_x, x), dim=-1)

      #forget gate
      self.f_t = torch.sigmoid(self.in_x @ self.W_f + self.b_f)
      self.c *= self.f_t

      #input gate
      self.i_t = torch.sigmoid(self.in_x @ self.W_i + self.b_i)
      self.c_t = torch.tanh(self.in_x @ self.W_c + self.b_c)
      self.i_prod = self.i_t * self.c_t
      self.c += self.i_prod

      #output gate
      self.o_t = torch.sigmoid(self.in_x @ self.W_o + self.b_o)
      self.ogo = torch.tanh(self.c) * self.o_t
      self.h_x = self.ogo
      long_term_outputs.append(self.c)
      short_term_outputs.append(self.h_x)

    return long_term_outputs, short_term_outputs, self.c, self.h_x

  def parameters(self):
    return [self.W_f, self.b_f, self.W_i, self.b_i, self.W_c, self.b_c, self.W_o, self.b_o]

  def __repr__(self):
    return f"LSTM()"


In [51]:
class BiDirectionalLSTM():
  def __init__(self, batch_size, n_features, n_hidden):
    self.f_lstm = LSTMCell(batch_size, n_features, n_hidden)
    self.b_lstm = LSTMCell(batch_size, n_features, n_hidden)
    self.n_hidden = n_hidden * 2

  def __call__(self, x, state=(None, None)):
    (f_c, f_h), (b_c, b_h) = state if all(state) else (None, None), (None, None)

    x_reverse = torch.flip(x, [0])

    _, _, cf, hf = self.f_lstm(x, (f_c, f_h))
    _, _, cb, hb = self.b_lstm(x_reverse, (b_c, b_h))

    #might have to concatenate for the lists of hidden states; not sure yet.
    #concats are for the lists of prev. states
    return (cf, cb), (hf, hb)

  def __repr__(self):
    return f"BiDirectionalLSTM(n_hidden={self.n_hidden})"

  def parameters(self):
    return self.f_lstm.parameters() + self.b_lstm.parameters()





In [64]:
blstm = BiDirectionalLSTM(seqs.shape[1], 1, 10)
blinear = Linear(blstm.n_hidden, 1)


In [65]:
epochs = 100
params = blstm.parameters() + blinear.parameters()

for p in params:
  p.requires_grad = True

for i in range(epochs):
  #forward pass
  long_term, short_term = blstm(seqs)
  state = torch.cat(short_term, -1)
  logits = blinear(state)
  loss = F.mse_loss(logits, targets)
  print(loss.item())

  #backward pass
  for p in params:
    p.grad = None

  loss.backward()

  lr = 0.08
  for p in params:
    p.data += -lr*p.grad

#print(loss.item())


7728.30126953125
385.8446960449219
187.7572784423828
175.14674377441406
175.0102996826172
175.00881958007812
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00881958007812
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.00880432128906
175.0088043212890

In [ ]:
epochs = 100
params = brnn.parameters() + blinear.parameters()

for p in params:
  p.requires_grad = True

for i in range(epochs):
  #forward pass
  _, state = brnn(seqs)
  state = torch.cat(state, -1)
  logits = blinear(state)
  loss = F.mse_loss(logits, targets)
  print(loss)

  #backward pass
  for p in params:
    p.grad = None

  loss.backward()

  lr = 0.08
  for p in params:
    p.data += -lr*p.grad

#print(loss.item())


tensor(8248.3936, grad_fn=<MseLossBackward0>)
tensor(39250.1523, grad_fn=<MseLossBackward0>)
tensor(189298.3125, grad_fn=<MseLossBackward0>)
tensor(915532.5625, grad_fn=<MseLossBackward0>)
tensor(4430503.5000, grad_fn=<MseLossBackward0>)
tensor(21442982., grad_fn=<MseLossBackward0>)
tensor(1.0378e+08, grad_fn=<MseLossBackward0>)
tensor(5.0231e+08, grad_fn=<MseLossBackward0>)
tensor(2.4312e+09, grad_fn=<MseLossBackward0>)
tensor(1.1767e+10, grad_fn=<MseLossBackward0>)
tensor(5.6952e+10, grad_fn=<MseLossBackward0>)
tensor(2.7565e+11, grad_fn=<MseLossBackward0>)
tensor(1.3341e+12, grad_fn=<MseLossBackward0>)
tensor(6.4572e+12, grad_fn=<MseLossBackward0>)
tensor(3.1253e+13, grad_fn=<MseLossBackward0>)
tensor(1.5126e+14, grad_fn=<MseLossBackward0>)
tensor(7.3210e+14, grad_fn=<MseLossBackward0>)
tensor(3.5434e+15, grad_fn=<MseLossBackward0>)
tensor(1.7150e+16, grad_fn=<MseLossBackward0>)
tensor(8.3006e+16, grad_fn=<MseLossBackward0>)
tensor(4.0175e+17, grad_fn=<MseLossBackward0>)
tensor(1.94

In [ ]:
lstm = LSTMCell(seqs.shape[1], 1, 10)
lstm_linear = Linear(10, 1)
lstm_linear.weights *= 0.1

In [ ]:
epochs = 200
lstm_params = lstm.parameters() + lstm_linear.parameters()
for p in lstm_params:
  p.requires_grad = True

for i in range(epochs):
  #forward pass
  _, _, _, output = lstm(seqs)
  logits = lstm_linear(output)
  loss = F.mse_loss(logits, targets)
  print(loss.item())

  #backward pass
  for p in lstm_params:
    p.grad = None

  loss.backward(retain_graph=True)

  lr = 0.02
  for p in lstm_params:
    p.data += -lr*p.grad

#print(loss.item())


In [18]:
class GRUCell():
  def __init__(self, batch_size, n_features, n_hidden, s=1):
    self.n_hidden = n_hidden
    self.batch_size = batch_size
    self.n_features = n_features
    self.W_r = torch.randn((self.n_features + self.n_hidden, self.n_hidden)) * s
    self.W_z = torch.randn((self.n_features + self.n_hidden, self.n_hidden)) * s
    self.W_h = torch.randn((self.n_features + self.n_hidden, self.n_hidden)) * s
    self.b_h = torch.ones(self.n_hidden) * s

  def __call__(self, input, state=None):
    self.h_x = state or torch.zeros((self.batch_size, self.n_hidden))
    outputs = []
    for x in input:
      self.in_x = torch.concat((self.h_x, x), dim=-1)

      #reset gate
      self.r_t = torch.sigmoid(self.in_x @ self.W_r)

      #update gate
      self.z_t = torch.sigmoid(self.in_x @ self.W_z)

      #candidate hidden state
      self.had = self.r_t * self.h_x
      self.inter = torch.concat((self.had, x), dim=-1)
      self.h_t = torch.tanh(self.inter @ self.W_h + self.b_h)

      #hidden state
      self.h_x = (
          (1 - self.z_t) * self.h_x + self.z_t * self.h_t)
      outputs.append(self.h_x)

    return outputs, self.h_x

  def __repr__(self):
    return f"GRU(n_hidden={self.n_hidden})"

  def parameters(self):
    return [self.W_r, self.W_z, self.W_h, self.b_h]


In [25]:
class BiDirectionalGRU():
  def __init__(self, batch_size, n_features, n_hidden):
    self.f_gru = GRUCell(batch_size, n_features, n_hidden)
    self.b_gru = GRUCell(batch_size, n_features, n_hidden)
    self.n_hidden = n_hidden * 2

  def __call__(self, x, state=(None, None)):
    f_h, b_h = state if all(state) else (None, None)
    _, output_f = self.f_gru(x, f_h)
    x_rev = torch.flip(x, [0])
    #would also have to check if the state should also be reversed before being
    #being fed to the model. As of now, it isn't
    _, output_b = self.b_gru(x_rev, b_h)
    return (output_f, output_b)

  def __repr__(self):
    return f"BiDirectionalGRU(n_hidden={self.n_hidden})"

  def parameters(self):
    return self.f_gru.parameters() + self.b_gru.parameters()


In [35]:
gru = BiDirectionalGRU(seqs.shape[1], 1, 40)
gru_linear = Linear(gru.n_hidden, 1)
gru_linear.weights *= 0.1

In [36]:
epochs = 200
gru_params = gru.parameters() + gru_linear.parameters()
for p in gru_params:
  p.requires_grad = True


for i in range(epochs):
  #forward pass
  output_f, output_b = gru(seqs)
  output = torch.concat((output_f, output_b), -1)
  logits = gru_linear(output)
  loss = F.mse_loss(logits, targets)
  print(loss.item())


  #backward pass
  for p in gru_params:
    p.grad = None

  loss.backward()

  lr = 0.02
  for p in gru_params:
    p.data += -lr*p.grad
#print(loss.item())


8425.806640625
6054.2998046875
5151.31494140625
4387.076171875
3740.105224609375
3192.50244140625
2729.01708984375
2336.718017578125
2004.67578125
1723.635498046875
1485.7669677734375
1284.43408203125
1114.0245361328125
969.79296875
847.7133178710938
744.3855590820312
656.9302368164062
582.9072265625
520.2535400390625
467.2238464355469
422.33978271484375
384.3497009277344
352.1949157714844
324.97918701171875
301.9438171386719
282.4466857910156
265.94451904296875
251.97669982910156
240.1542510986328
230.1479034423828
221.678466796875
214.50999450683594
208.4425506591797
203.30714416503906
198.96060180664062
195.28167724609375
192.16775512695312
189.53219604492188
187.30145263671875
185.41331481933594
183.81515502929688
182.46253967285156
181.3175811767578
180.34857177734375
179.5284881591797
178.83428955078125
178.24671936035156
177.7493896484375
177.32852172851562
176.9722442626953
176.67068481445312
176.41549682617188
176.19949340820312
176.01661682128906
175.86181640625
175.730865478